# MTM settings recommendation, Phase 1: scoring the recipe grid on NSTX_MTM

Does the extent rule with an envelope margin `s` and a basis floor `F` (the "MTM recipe") reach `m11_new`'s growth-rate accuracy at lower cost, and does the floor fix the shortest-extent cases? Grid: `s` ∈ {1.0, 1.41, 2.0} × NXGRID floor `F` ∈ {14, 18, 22, 26}, `NBASIS = NXGRID − 6`, so the NBASIS floor is `F − 6`. Arms beside the grid: `m3_new`, `m5_new`, `m7_new`, `m9_new`, `m11_new`, and `gftm_fw174` (GFTM default `WIDTH = 1.74`; the NXGRID it actually ran is **24**, per `Fusion_PhD-lgp2`). Reference: GS2 on the NSTX_MTM Latin hypercubes `beta_q_shat_ky_n300` (tuning draw) and `beta_q_shat_ky_n1000` (confirmation draw). The two draws are never pooled.

## Pre-registered selection rule (committed before any cell of the grid was scored)

**Metric.** `medlog` = median over GS2-unstable cases ($\gamma_{GS2} > 10^{-3}\,c_s/a$) of $|\log_{10}(\gamma_{GFTM}/\gamma_{GS2})|$, using GFTM's **dominant** mode (`argmax` of `growth_rate` over `mode`, never `isel(mode=0)`). **Misses are kept, never dropped**: a case where the arm returns no mode with $\gamma > 0$, or has no deck at all (the rule's infeasible cases), scores $+\infty$. Every arm is scored on the same cases. Uncertainty: 95% case-bootstrap intervals (2000 resamples, the same resampled cases for every arm, fixed seed), including the paired difference to `m11_new`; McNemar on tearing-parity capture.

**Rule.** Tune on **n300**. Choose the cheapest `(s, F)` whose n300 medlog is within 0.02 of `m11_new`'s (medlog ≤ medlog(`m11_new`) + 0.02) **and** whose tearing-parity capture is ≥ `m11_new`'s. Report that cell on **n1000** with no re-tuning; if it fails the same test on n1000, that is reported as a failure. *Cheapest* is mean `NBASIS` actually used over the scored cases. *(Agent-added, not the user's: `NBASIS` does not depend on `s` at fixed `F`, so ties in cost are broken by the lower n300 medlog.)*

**Status of the rule in this notebook (user decision, 2026-09-24).** GFTM tearing-parity capture cannot be measured yet: pyrokinetics' `FieldLine.compute_linear_tearing_parameter` needs a theta-resolved `apar`, which a GFTM `gk_output` does not carry (pyrokinetics issue #594, open). The capture half of the rule, and McNemar with it, is **deferred**, so **the recipe is not frozen here**. This notebook reports the accuracy half — which cells come within 0.02 of `m11_new` on n300, and their n1000 values — as **provisional**. GS2's own mode is classified with that diagnostic, which does work on GS2, to split the population into GS2-tearing and other.

**Hypothesis (the user's).** At short extent the rule lacks basis functions. Q1 = cases whose one-sided extent under the `s = 1.41`, `F = 14` arm ($\mathrm{WIDTH}\cdot x_{max}(\mathrm{NXGRID})$) is below the 25th percentile of the GS2-unstable cases, per draw (the `t3ge`/`1c88` definition, a property of the case applied identically to every arm). Verdict, on both draws, at each `s`: **yes** if raising `F` makes the Q1 medlog difference to `m11_new` have a 95% interval that includes or lies below zero, while the Q2–Q4 medlog change from `F = 14` to `F = 26` has an interval that includes zero; **partial** if the Q1 gap narrows by at least half its `F = 14` value but one of those conditions fails; **no** otherwise.


## Imports and settings

Set `GK_DATA_ROOT` in `local.env` to the directory containing `GS2/` and `GFTM/`. Every arm is a leaf of scan information under the same project, case and draw. The grid leaves at `F = 14` are the pre-existing extent-rule arms (no `_f14` suffix); the others were built by `Fusion_PhD-bhx7.1`.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from pyrokinetics import Pyro, PyroHypercube
from pyrokinetics.diagnostics.field_line import FieldLine

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "mtm_settings_recommendation"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
run_template = "Runs"
project = "LATIN_HYPERCUBE"
case = "NSTX_MTM"
draws = {"n300": "beta_q_shat_ky_n300", "n1000": "beta_q_shat_ky_n1000"}  # tune, confirm
reference = ("GS2", "pyro_cube_avg")  # (code, scan information below the draw)
gftm_output = "pyro_cube"
metadata_file = "pyroscan.json"
output_file = "cube.nc"

scales = [1.0, 1.41, 2.0]  # envelope margin s
floors = [14, 18, 22, 26]  # NXGRID floor F; NBASIS floor F - 6
f14_leaf = {1.0: "extent_rule_tf035", 1.41: "extent_rule_tf035_s141", 2.0: "extent_rule_tf035_s200"}
leaves = {(s, F): f14_leaf[s] if F == 14 else f"extent_rule_tf035_s{round(100 * s):03d}_f{F}"
          for s in scales for F in floors}
leaves |= {"m3_new": "m3_new", "m5_new": "m5_new", "m7_new": "m7_new", "m9_new": "m9_new",
           "m11_new": "m11_new", "fw174": "gftm_fw174"}
grid = [(s, F) for s in scales for F in floors]

gs2_unstable = 1e-3  # c_s/a; GS2 cases at or below this are not scored
tearing_threshold = 0.5  # FieldLine T above this is GS2 tearing parity (even A_par)
margin = 0.02  # pre-registered medlog margin to m11_new
q1_arm, q1_percentile = (1.41, 14), 25  # extent quartile definition (t3ge / 1c88)
n_boot, seed = 2000, 0
n_extent_bins = 8

## Load data

Pyrokinetics loads each hypercube with its base input and GK output: one GS2 reference and 18 GFTM arms per draw.

In [ ]:
def load(code, *scan_information):
    directory = data_root.joinpath(code, run_template, project, case, *scan_information)
    cube = PyroHypercube(pyroscan_json=directory / metadata_file, load_base_pyro=True)
    cube.from_netcdf(directory / output_file)
    return cube

gs2_cubes = {draw: load(reference[0], scan, reference[1]) for draw, scan in draws.items()}
cubes = {(draw, label): load("GFTM", scan, leaf, gftm_output)
         for draw, scan in draws.items() for label, leaf in leaves.items()}
for draw in draws:
    print(draw, "GS2", dict(gs2_cubes[draw].gk_output.data.sizes),
          "GFTM samples", sorted({cubes[draw, l].gk_output.data.sizes["sample"] for l in leaves}))

## Calculate: growth-rate error per case

Arms are matched to GS2 by `sample_name`, never by position. The dominant mode is the `argmax` of `growth_rate` over `mode`. A case missing from an arm's cube (the extent rule's infeasible cases, which have no deck) or with no mode at $\gamma > 0$ is a **miss** and scores $+\infty$; it is never dropped. `NBASIS` is the value GFTM used (`nbasis_used`, from `out.gftm.localdump`) where the cube carries it; the fixed `m*_new` arms carry it only in their base deck, `NBASIS_MAX`, which every case of those arms shares.

The case extent that defines the quartiles is the one-sided outermost Gauss–Hermite node of the `s = 1.41`, `F = 14` arm, $\mathrm{WIDTH}\cdot x_{max}$, where $x_{max}$ is the largest root of GFTM's $2\,\mathrm{NXGRID}-1$-point Hermite rule (`numpy`'s `hermgauss` gives the same roots). It is checked against the cube's own `achieved_extent_pi`. It is never taken from `gk_output`'s `theta`, which is a common output grid.

In [ ]:
names, gamma_gs2, unstable, gamma, nbasis, err, extent = {}, {}, {}, {}, {}, {}, {}
for draw in draws:
    gs2 = gs2_cubes[draw].gk_output.data
    names[draw] = [str(n) for n in gs2.sample_name.values]
    gamma_gs2[draw] = np.asarray(gs2.growth_rate.values, float)
    unstable[draw] = gamma_gs2[draw] > gs2_unstable
    for label in leaves:
        ds = cubes[draw, label].gk_output.data
        row = {str(n): i for i, n in enumerate(ds.sample_name.values)}
        take = np.array([row.get(n, -1) for n in names[draw]])  # -1: no deck for this case
        g = np.asarray(ds.growth_rate.transpose("sample", "mode").values, float)
        g = np.where(np.isfinite(g), g, -np.inf).max(axis=1)  # dominant mode
        gamma[draw, label] = np.where(take >= 0, g[take], np.nan)
        nb = (ds.nbasis_used.values if "nbasis_used" in ds.coords
              else np.full(ds.sizes["sample"], cubes[draw, label].base_pyro.gk_input.data["nbasis_max"]))
        nbasis[draw, label] = np.where(take >= 0, np.asarray(nb, float)[take], np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            e = np.abs(np.log10(gamma[draw, label] / gamma_gs2[draw]))
        err[draw, label] = np.where(gamma[draw, label] > 0, e, np.inf)

    ds = cubes[draw, q1_arm].gk_output.data
    n = ds.nxgrid_used.values.astype(int)
    x_max = {k: np.polynomial.hermite.hermgauss(2 * k - 1)[0].max() for k in np.unique(n)}
    e = ds.width_used.values * np.array([x_max[k] for k in n]) / np.pi
    assert np.allclose(e, ds.achieved_extent_pi.values, rtol=1e-5), "extent disagrees with the cube"
    row = {str(s): i for i, s in enumerate(ds.sample_name.values)}
    extent[draw] = np.array([e[row[s]] if s in row else np.nan for s in names[draw]])
    print(f"{draw}: GS2-unstable {unstable[draw].sum()} of {len(names[draw])}; "
          f"no extent (infeasible) among them {np.isnan(extent[draw][unstable[draw]]).sum()}")
    print("  misses:", {str(l): int(np.isinf(err[draw, l][unstable[draw]]).sum()) for l in leaves})

Reference check against `Fusion_PhD-1c88` (`results/mtm_extent_rule_validate/README.md`, "Full population"): on its strictly paired population — GS2-unstable cases where the extent rule `s = 1`, `s = 1.41`, `m3_new`…`m11_new` and `fw174` all return a mode — it reports medlog n300 / n1000 of `m11_new` 0.254 / 0.281, `s = 1.41` 0.342 / 0.310, `fw174` 0.274 / 0.318. The same numbers should come out here before the new definition (misses kept, all GS2-unstable cases) is used.

In [ ]:
old_arms = [(1.0, 14), (1.41, 14), "m3_new", "m5_new", "m7_new", "m9_new", "m11_new", "fw174"]
for draw in draws:
    paired = unstable[draw] & np.all([np.isfinite(err[draw, l]) for l in old_arms], axis=0)
    print(draw, "paired", paired.sum(),
          {str(l): round(float(np.median(err[draw, l][paired])), 3) for l in ((1.41, 14), "m11_new", "fw174")})

## Classify the GS2 mode

pyrokinetics' `FieldLine.compute_linear_tearing_parameter` gives $T = \left|\int A_\parallel \sqrt{g_{\theta\theta}}\,\mathrm{d}\theta\right| / \int |A_\parallel \sqrt{g_{\theta\theta}}|\,\mathrm{d}\theta$, $T \to 1$ for even (tearing-parity) $A_\parallel$. It takes a `Pyro`, so each GS2-unstable case is read from its own run directory, whose deck carries the geometry that case ran (the per-sample Pyros of `build_pyro_dict()` carry the base geometry; `Fusion_PhD-35kz.19`). GS2 has one mode per case. It is used only to split the scored population; the GFTM arms cannot be classified (pyrokinetics #594). This cell reads ~1150 GS2 outputs and takes roughly half an hour.

In [ ]:
tearing = {}
for draw in draws:
    runs = Path(gs2_cubes[draw].base_directory)
    T = np.full(len(names[draw]), np.nan)
    for i in np.flatnonzero(unstable[draw]):
        pyro = Pyro(gk_file=runs / names[draw][i] / gs2_cubes[draw].file_name)
        pyro.load_gk_output(load_fluxes=False, load_moments=False)
        T[i] = FieldLine(pyro).compute_linear_tearing_parameter()
    assert np.isfinite(T[unstable[draw]]).all(), "diagnostic returned a non-finite T"
    tearing[draw] = T > tearing_threshold
    print(f"{draw}: GS2 tearing parity {tearing[draw][unstable[draw]].mean():.3f} of {unstable[draw].sum()}")

## Score every arm, with bootstrap intervals

Populations per draw: all GS2-unstable cases; the extent quartile Q1 and the rest (Q2–Q4) — cases with no extent (infeasible for the rule) fall in neither, but stay in "all"; and the GS2 tearing / other split. Within a population the same bootstrap resamples are used for every arm, so differences to `m11_new` (and between grid cells) are paired.

In [ ]:
rng = np.random.default_rng(seed)
table = {}
for draw in draws:
    q25 = np.nanpercentile(extent[draw][unstable[draw]], q1_percentile)
    populations = {"all": unstable[draw], "Q1": unstable[draw] & (extent[draw] < q25),
                   "Q2-Q4": unstable[draw] & (extent[draw] >= q25),
                   "GS2 tearing": unstable[draw] & tearing[draw], "GS2 other": unstable[draw] & ~tearing[draw]}
    print(f"{draw}: Q1 upper extent {q25:.3f} pi;", {p: int(m.sum()) for p, m in populations.items()})
    for pop, m in populations.items():
        b = rng.integers(0, m.sum(), (n_boot, m.sum()))
        for label in leaves:
            e = err[draw, label][m]
            table[draw, pop, label] = dict(n=int(m.sum()), medlog=np.median(e), boot=np.median(e[b], axis=1),
                                           misses=int(np.isinf(e).sum()), nbasis=np.nanmean(nbasis[draw, label][m]))
        ref = table[draw, pop, "m11_new"]
        for label in leaves:
            r = table[draw, pop, label]
            r["ci"] = np.percentile(r["boot"], [2.5, 97.5])
            r["delta"] = r["medlog"] - ref["medlog"]
            r["delta_ci"] = np.percentile(r["boot"] - ref["boot"], [2.5, 97.5])

for pop in ("all", "Q1", "Q2-Q4", "GS2 tearing", "GS2 other"):
    print(f"\n{pop}: medlog [95% CI] | minus m11_new [95% CI] | misses | mean NBASIS")
    for label in leaves:
        print(f"  {str(label):12s}", "   ".join(
            f"{d} {r['medlog']:.3f} [{r['ci'][0]:.3f},{r['ci'][1]:.3f}] | {r['delta']:+.3f} "
            f"[{r['delta_ci'][0]:+.3f},{r['delta_ci'][1]:+.3f}] | {r['misses']:3d} | {r['nbasis']:5.1f}"
            for d in draws for r in [table[d, pop, label]]))

## Apply the accuracy half of the pre-registered rule

On n300: the grid cells within `margin` of `m11_new`'s medlog, ordered by cost (mean `NBASIS`, then medlog). The cheapest passing cell is the **provisional** recipe; it is reported on n1000 against the same test, not re-tuned. The capture half is deferred (see the first cell), so nothing is frozen.

In [ ]:
bar = {d: table[d, "all", "m11_new"]["medlog"] + margin for d in draws}
passing = [c for c in grid if table["n300", "all", c]["medlog"] <= bar["n300"]]
passing.sort(key=lambda c: (table["n300", "all", c]["nbasis"], table["n300", "all", c]["medlog"]))
provisional = passing[0] if passing else None
print(f"bar: m11_new medlog + {margin} = n300 {bar['n300']:.3f}, n1000 {bar['n1000']:.3f}")
for c in passing:
    r3, r1 = table["n300", "all", c], table["n1000", "all", c]
    print(f"  pass n300 {c}: mean NBASIS {r3['nbasis']:.1f}, medlog n300 {r3['medlog']:.3f}, "
          f"n1000 {r1['medlog']:.3f} -> {'holds' if r1['medlog'] <= bar['n1000'] else 'FAILS'} on n1000")
print("provisional recipe (accuracy half only):", provisional)

## Hypothesis: does the floor close the Q1 gap to `m11_new` without moving Q2–Q4?

For each draw and `s`: the Q1 medlog difference to `m11_new` at each floor, and the change in Q2–Q4 medlog from `F = 14` to `F = 26` (paired bootstrap interval).

In [ ]:
for draw in draws:
    for s in scales:
        q1 = [table[draw, "Q1", (s, F)] for F in floors]
        lo, hi = (table[draw, "Q2-Q4", (s, F)] for F in (floors[0], floors[-1]))
        change = hi["medlog"] - lo["medlog"]
        change_ci = np.percentile(hi["boot"] - lo["boot"], [2.5, 97.5])
        print(f"{draw} s={s}: Q1 minus m11_new by F", "  ".join(
            f"{F}: {r['delta']:+.3f} [{r['delta_ci'][0]:+.3f},{r['delta_ci'][1]:+.3f}]" for F, r in zip(floors, q1)),
              f"| Q2-Q4 F{floors[0]}->F{floors[-1]}: {change:+.3f} [{change_ci[0]:+.3f},{change_ci[1]:+.3f}]")

## Plot

Figure 4: n300 medlog over the grid, n1000 in brackets, `m11_new` in the title. Dashed outline: within the pre-registered margin on n300; solid box: the provisional recipe.

In [ ]:
M = np.array([[table["n300", "all", (s, F)]["medlog"] for F in floors] for s in scales])
fig4, ax = plt.subplots(figsize=(7, 4.2))
im = ax.imshow(M, origin="lower", cmap="YlOrRd", aspect="auto")
ax.grid(False)
for i, s in enumerate(scales):
    for j, F in enumerate(floors):
        ax.text(j, i, f"{M[i, j]:.3f}\n({table['n1000', 'all', (s, F)]['medlog']:.3f})", ha="center", va="center")
        if (s, F) in passing:
            ax.add_patch(plt.Rectangle((j - 0.45, i - 0.45), 0.9, 0.9, fill=False, ls="--", lw=1.5))
        if (s, F) == provisional:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, lw=3))
ax.set(xticks=range(len(floors)), xticklabels=[f"{F} ({F - 6})" for F in floors], yticks=range(len(scales)),
       yticklabels=scales, xlabel="NXGRID floor F (NBASIS floor)", ylabel="envelope margin s",
       title=f"medlog n300 (n1000); m11_new {table['n300', 'all', 'm11_new']['medlog']:.3f} "
             f"({table['n1000', 'all', 'm11_new']['medlog']:.3f})")
fig4.colorbar(im, label="n300 medlog")
plt.show()

Figure 5: medlog against case extent, in `n_extent_bins` equal-count bins of the GS2-unstable cases per draw, with 95% bootstrap bands; Q1 shaded. Arms: the old rule (`s = 1`, `F = 14`), the provisional recipe and its `s` at the highest floor, `m9_new`, `m11_new`, `fw174`.

In [ ]:
recipe = provisional or q1_arm
fig5_arms = list(dict.fromkeys([(1.0, 14), recipe, (recipe[0], floors[-1]), "m9_new", "m11_new", "fw174"]))
fig5, axes = plt.subplots(1, 2, sharey=True, figsize=(11, 4.5))
for ax, draw in zip(axes, draws):
    x = extent[draw]
    edges = np.nanpercentile(x[unstable[draw]], np.linspace(0, 100, n_extent_bins + 1))
    bins = [unstable[draw] & (x >= a) & (x <= b) for a, b in zip(edges[:-1], edges[1:])]
    centres = [np.median(x[m]) for m in bins]
    for label in fig5_arms:
        med = [np.median(err[draw, label][m]) for m in bins]
        ci = np.array([np.percentile(np.median(err[draw, label][m][rng.integers(0, m.sum(), (n_boot, m.sum()))], axis=1),
                                     [2.5, 97.5]) for m in bins])
        line, = ax.plot(centres, med, label=f"rule s={label[0]}, F={label[1]}" if isinstance(label, tuple) else label)
        ax.fill_between(centres, ci[:, 0], ci[:, 1], color=line.get_color(), alpha=0.15)
    ax.axvspan(edges[0], np.nanpercentile(x[unstable[draw]], q1_percentile), color="grey", alpha=0.2, label="Q1")
    ax.set(xscale="log", xlabel=r"case extent, $s=1.41$ arm [$\pi$, one-sided]", title=draw)
axes[0].set_ylabel(r"medlog $|\log_{10}(\gamma_{GFTM}/\gamma_{GS2})|$")
axes[1].legend(fontsize=8)
plt.show()

Figure 6 (stands in for capture vs medlog until pyrokinetics #594 is fixed): medlog against mean `NBASIS` used, per arm, both draws, 95% bootstrap bars. The dotted lines mark each draw's `m11_new` + `margin`.

In [ ]:
fig6, ax = plt.subplots(figsize=(8, 5.5))
styles = {"n300": dict(marker="o"), "n1000": dict(marker="s", mfc="none")}
colours = {1.0: "C0", 1.41: "C1", 2.0: "C2"}
for draw, style in styles.items():
    for label in leaves:
        r = table[draw, "all", label]
        colour = colours[label[0]] if isinstance(label, tuple) else "C3" if label == "fw174" else "C7"
        ax.errorbar(r["nbasis"], r["medlog"], yerr=np.abs(r["ci"] - r["medlog"])[:, None], color=colour, ls="", **style)
        if not isinstance(label, tuple) and draw == "n300":
            ax.annotate(label, (r["nbasis"], r["medlog"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.axhline(bar[draw], color="k", ls=":", lw=1, marker="")
handles = [plt.Line2D([], [], color=c, ls="", marker="o", label=f"rule s={s}, F=14..26") for s, c in colours.items()]
handles += [plt.Line2D([], [], color="k", ls="", **st, label=d) for d, st in styles.items()]
ax.legend(handles=handles, fontsize=8)
ax.set(xlabel="mean NBASIS used (cost)", ylabel="medlog (all GS2-unstable, misses kept)")
plt.show()

## Save

Replaces the same filenames.

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
fig4.savefig(output_dir / "fig4_grid_medlog.png")
fig5.savefig(output_dir / "fig5_medlog_vs_extent.png")
fig6.savefig(output_dir / "fig6_medlog_vs_nbasis.png")

## Interpretation

To be written from the executed results.